In [80]:
import re
import subprocess
from pathlib import Path
import pymupdf
from memvid_sdk import create, use
import pymupdf4llm
from shared import embedder


In [ ]:
import re
import subprocess
from pathlib import Path


from shared import embedder

DOCS_DIR = Path("../documents")


HEADING_RE = re.compile(r"^\s*(\d+(?:\.\d+)*)\.?\s+(\S.*)$", re.MULTILINE)
# Table-of-contents lines look like real headings but trail off in dot leaders
# and a page number (e.g. "1.2 Профиль компании ....... 6") - skip those.
TOC_LINE_RE = re.compile(r"\.{3,}\s*\d+\s*$")
# Bare figure captions ("Рисунок 3 – ...") don't carry standalone meaning -
# they get folded into the paragraph they follow instead of chunked alone.
FIGURE_CAPTION_RE = re.compile(r"^Рисунок\s+\d+\s*[–-]")

# mxbai-embed-large's Ollama context window is 512 tokens; Cyrillic text runs
# well under 1 char/token, so keep chunks well short of that to avoid MV_HTTP 500s.
CHUNK_SIZE = 400


def extract_text(pdf_path: Path) -> str:
    # pypdf mangles this PDF's font encoding (injects spurious mid-word
    # spaces) and pymupdf can't even parse it ("invalid key in dict"); the
    # system's poppler (pdftotext) extracts it cleanly, so shell out to it.
    result = subprocess.run(
        ["pdftotext", str(pdf_path), "-"],
        capture_output=True,
        check=True,
    )
    return result.stdout.decode("utf-8")


def clean_text(text: str) -> str:
    text = text.replace("\x0c", "\n")  # page-break marker -> plain newline
    # Bullet lists use a custom font glyph that lands in the Private Use Area;
    # it always sits at the start of a bulleted line, so normalize it to "-".
    text = re.sub(r"(?m)^[-]\s*", "- ", text)
    text = re.sub(r"[-]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text


def split_into_units(body: str) -> list[str]:
    """Split a section body into paragraph/bullet units, never mid-sentence.

    Each block (blank-line-delimited) is further split so every "- " bullet
    becomes its own unit, instead of the whole bullet list being one giant
    unit that pack_units() would later have to hard-wrap mid-bullet."""
    blocks = [b for b in re.split(r"\n\s*\n", body) if b.strip()]
    units: list[str] = []
    for block in blocks:
        lines = [line.strip() for line in block.splitlines() if line.strip()]
        segments: list[list[str]] = []
        for line in lines:
            if not segments or line.startswith("- "):
                segments.append([line])
            else:
                segments[-1].append(line)
        for segment in segments:
            joined = " ".join(segment)
            if units and FIGURE_CAPTION_RE.match(joined):
                units[-1] = f"{units[-1]} {joined}"
            else:
                units.append(joined)
    return units


def pack_units(units: list[str], max_chars: int = CHUNK_SIZE) -> list[str]:
    """Greedily pack whole units into chunks, carrying the last unit of a
    chunk forward into the next one for context continuity. Only a single
    unit that alone exceeds the budget gets hard-wrapped, and only on
    whitespace (never mid-word)."""
    chunks: list[str] = []
    current: list[str] = []
    current_len = 0
    i = 0
    while i < len(units):
        unit = units[i]
        if len(unit) > max_chars:
            if current:
                chunks.append(" ".join(current))
                current, current_len = [], 0
            piece = ""
            for word in unit.split(" "):
                candidate = f"{piece} {word}".strip()
                if piece and len(candidate) > max_chars:
                    chunks.append(piece)
                    piece = word
                else:
                    piece = candidate
            if piece:
                chunks.append(piece)
            i += 1
            continue

        candidate_len = current_len + (1 if current else 0) + len(unit)
        if current and candidate_len > max_chars:
            chunks.append(" ".join(current))
            # Soft overlap: carry the last unit forward for continuity, but
            # only if it still leaves room for the unit we're about to add -
            # otherwise drop the overlap so we always make forward progress.
            overlap = current[-1:]
            if len(overlap[0]) + 1 + len(unit) <= max_chars:
                current, current_len = overlap, len(overlap[0])
            else:
                current, current_len = [], 0
            continue

        current.append(unit)
        current_len = candidate_len
        i += 1

    if current:
        chunks.append(" ".join(current))
    return chunks


def split_sections(text: str, source: str) -> list[dict]:
    headings = [m for m in HEADING_RE.finditer(text) if not TOC_LINE_RE.search(m.group(0))]
    if not headings:
        return [
            {
                "title": f"{source} (chunk {i + 1})",
                "text": chunk,
                "label": "manual",
                "labels": ["manual"],
                "metadata": {"source": source, "chunk": i + 1},
            }
            for i, chunk in enumerate(pack_units(split_into_units(text)))
        ]

    items = []
    for i, match in enumerate(headings):
        section_number = match.group(1)
        heading_text = match.group(2).strip()
        body_start = match.end()
        body_end = headings[i + 1].start() if i + 1 < len(headings) else len(text)
        body = text[body_start:body_end].strip()
        if not body:
            continue
        sub_chunks = pack_units(split_into_units(body))
        for j, chunk in enumerate(sub_chunks):
            title = f"{section_number} {heading_text}"
            if len(sub_chunks) > 1:
                title = f"{title} (part {j + 1})"
            items.append(
                {
                    "title": title,
                    "text": chunk,
                    "label": "manual",
                    "labels": ["manual"],
                    "metadata": {
                        "source": source,
                        "section": section_number,
                        "section_title": heading_text,
                    },
                }
            )
    return items


In [22]:
import re
# mxbai-embed-large's Ollama context window is 512 tokens; Cyrillic text runs
# well under 1 char/token, so keep chunks well short of that to avoid MV_HTTP 500s.
CHUNK_SIZE = 400
TOC_LINE_RE = re.compile(r"\.{3,}\s*\d+\s*$")
HEADING_RE = re.compile(r"^\s*(\d+(?:\.\d+)*)\.?\s+(\S.*)$", re.MULTILINE)



In [165]:
STORE_PATH = Path("ingest.ipynb").absolute().parent.parent / 'data' / 'knowledge.mv2'
STORE_PATH

PosixPath('/home/bevzd/workspace/hackathon-tenderhack-nn-innosport/backend/data/knowledge.mv2')

In [166]:
# STORE_PATH = Path("ingest.ipynb").absolute().parent.parent
mem = use(
    "basic",
    str(STORE_PATH.absolute()),
    enable_lex=True,
    enable_vec=True,
)


In [143]:

def embed(objs):
    embeddings = [[float(x) for x in vec] for vec in embedder.embed_documents([item["text"] for item in objs])]


# put_many(embeddings=[...]) in memvid-sdk 2.0.160 only persists the vector
# index correctly for the LAST item when given a multi-item batch (all
# earlier items' vectors are silently dropped/overwritten). Calling it once
# per item avoids this and keeps every frame's vector correctly indexed.
    for item, embedding in zip(objs, embeddings):
        mem.put_many([item], embeddings=[embedding])


def save_to_memory(document, document_name):
    objs = split_sections(document, document_name)
    embeddings = [[float(x) for x in vec] for vec in embedder.embed_documents([item["text"] for item in objs])]


    # put_many(embeddings=[...]) in memvid-sdk 2.0.160 only persists the vector
    # index correctly for the LAST item when given a multi-item batch (all
    # earlier items' vectors are silently dropped/overwritten). Calling it once
    # per item avoids this and keeps every frame's vector correctly indexed.
    for item, embedding in zip(objs, embeddings):
        mem.put_many([item], embeddings=[embedding])

In [ ]:
document_path = Path("./documents/Инструкция_по_работе_с_машиночитаемыми_доверенностями.pdf")


page_count = pymupdf.open(document_path).page_count
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(2, page_count)))


In [ ]:
save_to_memory(doc, document_path.name[:-4])

In [47]:
document_path = Path("./documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf")


page_count = pymupdf.open(document_path).page_count
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(4, page_count - 1)))


In [75]:
# save_to_memory(doc, document_path.name[:-4])
objs = split_sections(doc, document_path.name[:-4])



In [76]:
embeddings = [[float(x) for x in vec] for vec in embedder.embed_documents([item["text"] for item in objs])]


In [77]:
for item, embedding in zip(objs, embeddings):
    mem.put_many([item], embeddings=[embedding])

In [175]:
mem.find("НМЦК", mode="lex", embedder=embedder)

{'query': 'НМЦК',
 'hits': [{'frame_id': 789,
   'uri': 'mv2://frames/789',
   'title': 'НМЦК',
   'rank': 1,
   'score': 7.863811016082764,
   'matches': 1,
   'snippet': 'НМЦК — Начальная (максимальная) цена контракта (при осуществлении государственных и муниципальных закупок) title: НМЦК labels: glossary document: "Инструкция_по_работе_с_Порталом_для_заказчика" path: "documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf" source: "Инструкция_по_работе_с_Порталом_для_заказчика" term: "НМЦК"',
   'text': 'НМЦК — Начальная (максимальная) цена контракта (при осуществлении государственных и муниципальных закупок) title: НМЦК labels: glossary document: "Инструкция_по_работе_с_Порталом_для_заказчика" path: "documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf" source: "Инструкция_по_работе_с_Порталом_для_заказчика" term: "НМЦК"',
   'tags': [],
   'labels': ['glossary'],
   'track': None,
   'created_at': '2026-09-12T08:32:45Z',
   'content_dates': []},
  {'frame_id': 461,
   

In [132]:
import json

json_str = pymupdf4llm.to_json(document_path, pages=[3])
data = json.loads(json_str)
table = None
for page_num, page in enumerate(data.get("pages", [])):
    print(f"\nPage {page_num}")

    for block in page.get("boxes", []):
        if block["boxclass"] == "table":
            table = block['table']



Page 0


In [138]:
extract = table['extract'][1:]

In [139]:
extract

[['1.',
  '223-ФЗ',
  'Федеральный закон от 18 июля 2011 г. № 223-ФЗ «О закупках товаров, \nработ, услуг отдельными видами юридических лиц»'],
 ['2.',
  '44-ФЗ',
  'Федеральный закон от 5 апреля 2013 года № 44-ФЗ «О контрактной \nсистеме в сфере закупок товаров, работ, услуг для обеспечения \nгосударственных и муниципальных нужд»'],
 ['3.', 'АРМ', 'Автоматизированное рабочее место'],
 ['4.',
  'Виджет',
  'Компактный информационный блок, размещенный на странице сайта. \nОбычно содержит данные или сервис другого сайта.'],
 ['5.',
  'ЕИС',
  'Официальный сайт Единой информационной системы в сфере закупок \n(www.zakupki.gov.ru)'],
 ['6.', 'ИНН', 'Идентификационный номер налогоплательщика'],
 ['7.', 'ИСиР', 'Информационные системы и ресурсы'],
 ['8.',
  'Котировочная \nсессия',
  'Проведение мини-аукционов или мини-конкурсов в рамках комплекса \nзадач проведения закупок товаров, работ и услуг'],
 ['9.',
  'КПГЗ',
  'Классификатор предметов государственного заказа города Москвы'],
 ['10.', 

In [144]:
glossary_source = document_path  # the "Портал для поставщика" PDF this glossary table came from

term_items = [
     {
         "title": term.strip(),
         "text": f"{term.strip()} — {definition.strip()}",
         "label": "glossary",
         "labels": ["glossary"],
         "metadata": {
             "source": glossary_source.name[:-4],
             "document": glossary_source.name[:-4],
             "path": str(glossary_source),
             "term": term.strip(),
         },
     }
     for _num, term, definition in extract
 ]

embed(term_items)

In [158]:
mem.find("НМЦК", mode="lex")
# mem.find("НМЦК", mode="auto", embedder=embedder)

{'query': 'НМЦК',
 'hits': [{'frame_id': 789,
   'uri': 'mv2://frames/789',
   'title': 'НМЦК',
   'rank': 1,
   'score': 7.863811016082764,
   'matches': 1,
   'snippet': 'НМЦК — Начальная (максимальная) цена контракта (при осуществлении государственных и муниципальных закупок) title: НМЦК labels: glossary document: "Инструкция_по_работе_с_Порталом_для_заказчика" path: "documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf" source: "Инструкция_по_работе_с_Порталом_для_заказчика" term: "НМЦК"',
   'text': 'НМЦК — Начальная (максимальная) цена контракта (при осуществлении государственных и муниципальных закупок) title: НМЦК labels: glossary document: "Инструкция_по_работе_с_Порталом_для_заказчика" path: "documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf" source: "Инструкция_по_работе_с_Порталом_для_заказчика" term: "НМЦК"',
   'tags': [],
   'labels': ['glossary'],
   'track': None,
   'created_at': '2026-09-12T08:32:45Z',
   'content_dates': []},
  {'frame_id': 461,
   

In [ ]:
document_path = Path("./documents/Инструкция_по_работе_с_Порталом_для_поставщика.pdf")


page_count = pymupdf.open(document_path).page_count
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(5, 6)))


In [ ]:
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(6, page_count)))


In [ ]:
objs = split_sections(doc, document_path.name[:-4])


In [ ]:
embeddings = [[float(x) for x in vec] for vec in embedder.embed_documents([item["text"] for item in objs])]


In [ ]:
for item, embedding in zip(objs, embeddings):
    mem.put_many([item], embeddings=[embedding])
